# Annexe 10.1 : chaîner les prompts

- [Cours](#cours)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# Réécrite pour accepter une liste de messages de longueur quelconque
def get_completion(messages, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=messages
    )
    return message.content[0].text

---

## Cours

Comme dit l'adage, « écrire, c'est réécrire ». De fait, **Claude améliore souvent la précision de sa réponse quand on le lui demande** !

Il existe de nombreuses façons de demander à Claude de « réfléchir à nouveau ». Les formulations qui paraissent naturelles pour demander à quelqu'un de vérifier son travail fonctionnent en général aussi avec Claude. (Consultez notre [documentation sur le chaînage de prompts](https://docs.anthropic.com/claude/docs/chain-prompts) pour d'autres exemples d'usage.)

### Exemples

Dans cet exemple, nous demandons à Claude de proposer dix mots... mais un ou plusieurs d'entre eux n'existent pas.

In [ ]:
# Prompt initial
first_user = "Name ten words that all end with the exact letters 'ab'."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print(first_response)

**Demander à Claude de rendre sa réponse plus exacte** corrige l'erreur !

Ci-dessous, nous reprenons la réponse incorrecte de Claude et ajoutons un tour de conversation qui lui demande de corriger sa réponse précédente.

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

Mais Claude corrige-t-il sa réponse simplement parce que nous le lui demandons ? Et si la réponse de départ était déjà correcte ? Claude perdrait-il confiance ? Ici, nous plaçons une réponse correcte dans `first_response` et lui demandons de vérifier à nouveau.

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

Si vous relancez plusieurs fois le bloc ci-dessus, vous remarquerez que Claude conserve les mots la plupart du temps, mais les modifie encore de temps en temps alors qu'ils sont tous corrects. Comment l'éviter ? Comme au chapitre 8, en lui laissant une porte de sortie ! Essayons encore une fois.

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

Relancez plusieurs fois le code ci-dessus : vous verrez que Claude tient beaucoup mieux sa position.

Le chaînage de prompts permet aussi de **demander à Claude d'améliorer ses réponses**. Ci-dessous, nous lui demandons d'abord d'écrire une histoire, puis d'améliorer l'histoire écrite. Les goûts varient, mais beaucoup trouveront la seconde version meilleure.

Commençons par générer la première version de l'histoire.

In [ ]:
# Prompt initial
first_user = "Write a three-sentence short story about a girl who likes to run."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print(first_response)

Demandons maintenant à Claude d'améliorer son premier jet.

In [ ]:
second_user = "Make the story better."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

Cette forme de substitution est très puissante. Nous avons utilisé des variables pour transmettre des listes, des mots, des réponses précédentes de Claude, etc. Vous pouvez aussi **utiliser la substitution pour faire ce qu'on appelle du « function calling » : demander à Claude d'exécuter une fonction, puis récupérer le résultat de cette fonction pour demander à Claude d'aller plus loin avec**. Cela fonctionne comme n'importe quelle substitution. Nous y revenons dans l'annexe suivante.

Voici un dernier exemple dans lequel le résultat d'un appel à Claude est injecté dans un autre appel, plus long. Commençons par le premier prompt (qui pré-remplit cette fois la réponse de Claude).

In [ ]:
first_user = """Find all names from the below text:

"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too."""

prefill = "<names>"

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(first_response)

Passons cette liste de noms à un autre prompt.

In [ ]:
second_user = "Alphabetize the list."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

Maintenant que vous connaissez le chaînage de prompts, passez à l'annexe 10.2 pour apprendre à mettre en œuvre l'appel de fonctions à l'aide du chaînage.

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Prompt initial
first_user = "Name ten words that all end with the exact letters 'ab'."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Please find replacements for all 'words' that are not real words."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = "Name ten words that all end with the exact letters 'ab'."

first_response = """Here are 10 words that end with the letters 'ab':

1. Cab
2. Dab
3. Grab
4. Gab
5. Jab
6. Lab
7. Nab
8. Slab
9. Tab
10. Blab"""

second_user = "Please find replacements for all 'words' that are not real words. If all the words are real words, return the original list."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
# Prompt initial
first_user = "Write a three-sentence short story about a girl who likes to run."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print(first_response)

In [ ]:
second_user = "Make the story better."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))

In [ ]:
first_user = """Find all names from the below text:

"Hey, Jesse. It's me, Erin. I'm calling about the party that Joey is throwing tomorrow. Keisha said she would come and I think Mel will be there too."""

prefill = "<names>"

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill
    
    }
]

# Stocker et afficher la réponse de Claude
first_response = get_completion(messages)
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(first_response)

In [ ]:
second_user = "Alphabetize the list."

# Tableau des messages de l'API
messages = [
    {
        "role": "user",
        "content": first_user
    
    },
    {
        "role": "assistant",
        "content": prefill + "\n" + first_response
    
    },
    {
        "role": "user",
        "content": second_user
    
    }
]

# Afficher la réponse de Claude
print("------------------------ Full messsages array with variable substutions ------------------------")
print(messages)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(messages))